# xMIP plugin demo

This notebook shows the xMIP-derived Woodpecker plugin on a tiny synthetic CMIP6 dataset. The goal is not to call one large preprocessing function, but to show the same style of cleanup as a readable set of Woodpecker fixes composed by a fix plan.

In [ ]:
from importlib.resources import as_file, files

import numpy as np
import woodpecker_xmip_plugin  # noqa: F401 - imports plugin fixes for editable installs

import woodpecker
from woodpecker.testing import make_cmip6

Start from Woodpecker's deterministic CMIP6 test factory, then introduce a few xMIP-style issues: non-standard `i`/`j` axes, `longitude`/`latitude` coordinate names, negative longitudes, centimeter vertical units, and GFDL-CM4 metadata that has a known branch-time correction.

In [ ]:
dataset = make_cmip6(
    overrides={"source_id": "GFDL-CM4", "experiment_id": "historical"},
    seed=7,
)
dataset = dataset.isel(time=slice(0, 2), lat=slice(0, 2), lon=slice(0, 3))
dataset = dataset.rename({"lat": "j", "lon": "i"})
dataset = dataset.expand_dims({"lev": [0.0, 100.0]})
dataset["lev"].attrs["units"] = "centimeters"

longitude = np.broadcast_to(np.array([-10.0, 0.0, 10.0]), (dataset.sizes["j"], 3))
latitude = np.broadcast_to(
    np.asarray(dataset["j"].values)[:, None],
    (dataset.sizes["j"], dataset.sizes["i"]),
)
dataset["longitude"] = (("j", "i"), longitude)
dataset["latitude"] = (("j", "i"), latitude)

dataset

In [ ]:
{
    "dims": dict(dataset.sizes),
    "data_vars": list(dataset.data_vars),
    "coords": list(dataset.coords),
    "lev_units": dataset["lev"].attrs.get("units"),
    "longitude_min": float(dataset["longitude"].min()),
    "branch_time_in_parent": dataset.attrs.get("branch_time_in_parent"),
}

The plugin ships a plan named `xmip.cmip6_preprocessing`. It composes small fixes such as axis renaming, longitude normalization, coordinate unit normalization, bounds handling, and known metadata correction.

In [ ]:
plan_ref = files("woodpecker_xmip_plugin") / "plans" / "cmip6_preprocessing.yaml"

with as_file(plan_ref) as plan_path:
    print(plan_path.read_text())

Check the dataset against the plan. Because dry-run checks do not simulate earlier plan steps, this first report only contains fixes visible on the original dataset. In write mode, the plan applies the fixes sequentially.

In [ ]:
with as_file(plan_ref) as plan_path:
    findings = woodpecker.plan.check(
        dataset,
        plan_path,
        plan_id="xmip.cmip6_preprocessing",
    )

findings.fix_ids

In [ ]:
with as_file(plan_ref) as plan_path:
    preview = woodpecker.plan.fix(
        dataset,
        plan_path,
        plan_id="xmip.cmip6_preprocessing",
        dry_run=True,
    )

preview.stats

Apply the plan in memory and inspect the result.

In [ ]:
with as_file(plan_ref) as plan_path:
    result = woodpecker.plan.fix(
        dataset,
        plan_path,
        plan_id="xmip.cmip6_preprocessing",
        dry_run=False,
    )

result.stats

In [ ]:
{
    "dims": dict(dataset.sizes),
    "data_vars": list(dataset.data_vars),
    "coords": list(dataset.coords),
    "lev_values": dataset["lev"].values.tolist(),
    "lev_units": dataset["lev"].attrs.get("units"),
    "lon_min": float(dataset["lon"].min()),
    "branch_time_in_parent": dataset.attrs.get("branch_time_in_parent"),
}

In [ ]:
with as_file(plan_ref) as plan_path:
    woodpecker.plan.check(
        dataset,
        plan_path,
        plan_id="xmip.cmip6_preprocessing",
    )